# Create a Simple Reflex-Based Lunar Lander Agent

In this example, we will use Gymnasium, an environment to train agents via reinforcement learning (RL). We will not use RL here but just use the environment with a custom simple reflex-based agent. 

## Install Gymnasium

The documentation for Gymnasium is available at https://gymnasium.farama.org/ 

Steps:
1. Create a new folder and open it with VS Code and install all needed Python Extensions in VS Code.
2. Create a new virtual environment (CTRL-Shift P Python Create Environment...)
3. I needed to install swig and the Python C++ headers on WSL2 via the terminal
    * `sudo apt install swig`
    * `sudo apt-get install python3-dev` 
4. Install python libraries for gymnasium with the needed extras

In [19]:
%pip install -q "swig>=4,<5" "gymnasium[box2d,classic-control]>=1.0,<2"  # da bat san, chi can chay 1 lan roi restart kernel

Additional installs for screen capturing so environment visualizations can be converted to embedded videos.
For recording videos, I had to install
    
* `sudo apt-get install xvfb ffmpeg`

Additional python packages for screen capturing.

In [20]:
%pip install -q "pyvirtualdisplay>=3,<4" "gymnasium[other]>=1.0,<2"  # da bat san, chi can chay 1 lan roi restart kernel

## The Lunar Lander Environment 

![Luna Lander image](https://gymnasium.farama.org/_images/lunar_lander.gif)

The documentation of the environment is available at: https://gymnasium.farama.org/environments/box2d/lunar_lander/

* Performance Measure: A reward of -100 or +100 points for crashing or landing safely respectively. We do not use 
  intermediate rewards here.

* Environment: This environment is a classic rocket trajectory optimization problem. A ship needs to land safely. The space is **continuous** with
  x and y coordinates in the range [-2.5, 2.5]. The landing pad is at coordinate (0,0).

* Actuators:  According to Pontryagin’s
  maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off. There are four discrete actions available:

    - 0: do nothing
    - 1: fire left orientation engine
    - 2: fire main engine
    - 3: fire right orientation engine

* Sensors: Each observation is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

The different ranges/settings of the environment can be queried.

In [21]:
import gymnasium as gym
import numpy as np
np.set_printoptions(precision=2)

In [22]:
def query_environment(name):
    env = gym.make(name)
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {env.spec.max_episode_steps}")
    print(f"Nondeterministic: {env.spec.nondeterministic}")
   # print(f"Reward Range: {env.reward_range}")
    print(f"Reward Threshold: {env.spec.reward_threshold}")
    env.close()

query_environment("LunarLander-v3")

Action Space: Discrete(4)
Observation Space: Box([ -2.5   -2.5  -10.   -10.    -6.28 -10.    -0.    -0.  ], [ 2.5   2.5  10.   10.    6.28 10.    1.    1.  ], (8,), float32)
Max Episode Steps: 1000
Nondeterministic: False
Reward Threshold: 200



Gymnasium environments are implemented as classes with a `make` method to create the environment, a `reset` method, and a `step` method to execute an action.
To use it with an agent function that expects percepts and returns an action, we need write glue code that connects the environment with the agent function.

In [23]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    # run one episode
    for _ in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        if verbose:
            print (f"Obs: {observation} -> Action: {action}")

        # step: execute an action in the environment
        observation, reward, terminated, truncated, info = env.step(action)

        # render the environment
        if render:
            env.render()

        if terminated:
            if verbose:
                print(f"Final Reward: {reward}")
            break
    
    return reward

## Example: A Random Agent

We randomly return one of the actions. The environment accepts the integers 0-3.


In [24]:
def random_agent_function(observation): 
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return np.random.choice([0, 1, 2, 3], p=[0.25, 0.25, 0.25, 0.25])

Run an episode.

In [25]:
env = gym.make("LunarLander-v3", render_mode="human")

run_episode(random_agent_function, env)

env.close()

Obs: [-0.01  1.42 -0.6   0.46  0.01  0.14  0.    0.  ] -> Action: 0
Obs: [-0.01  1.43 -0.6   0.44  0.01  0.13  0.    0.  ] -> Action: 0
Obs: [-0.02  1.44 -0.6   0.41  0.02  0.13  0.    0.  ] -> Action: 1
Obs: [-0.02  1.45 -0.61  0.38  0.03  0.18  0.    0.  ] -> Action: 0
Obs: [-0.03  1.46 -0.61  0.36  0.04  0.18  0.    0.  ] -> Action: 3
Obs: [-0.04  1.46 -0.6   0.33  0.04  0.13  0.    0.  ] -> Action: 0
Obs: [-0.04  1.47 -0.6   0.31  0.05  0.13  0.    0.  ] -> Action: 2
Obs: [-0.05  1.48 -0.61  0.34  0.06  0.12  0.    0.  ] -> Action: 0
Obs: [-0.05  1.49 -0.61  0.31  0.06  0.12  0.    0.  ] -> Action: 1
Obs: [-0.06  1.49 -0.62  0.28  0.07  0.17  0.    0.  ] -> Action: 3
Obs: [-0.07  1.5  -0.61  0.26  0.08  0.12  0.    0.  ] -> Action: 2
Obs: [-0.07  1.5  -0.63  0.3   0.08  0.11  0.    0.  ] -> Action: 0
Obs: [-0.08  1.51 -0.63  0.27  0.09  0.11  0.    0.  ] -> Action: 1
Obs: [-0.08  1.52 -0.63  0.25  0.1   0.14  0.    0.  ] -> Action: 3
Obs: [-0.09  1.52 -0.62  0.22  0.1   0.1   0.   

Gymnasium displays environments using `render()` method on the local display. Headless installations like Google Colab do not have a display, but the output can be captured using a virtual display as a video and then add the video to the notebook.

We need to start a virtual display.

Now we can use wrappers for the environment to record the display. The functions are provided in
[display_record.py].



In [26]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Artificial_Intelligence/refs/heads/master/Agents/")

In [27]:
import gymnasium_display_recorder as dr

env = dr.gym_make('LunarLander-v3', 'LL1', render_fps=30)
run_episode(random_agent_function, env, verbose=False)
env.close()

dr.show('LL1')

## A Simple Reflex-Based Agent

To make the code easier to read, we use enumerations for actions (integers) and observations (index in the observation vector).

In [28]:
from enum import Enum

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7


Define a simple agent that uses the main thruster to reduce the falling speed if it gets too fast.

In [29]:
def rocket_agent_function(observation):
    """A simple agent function."""

    # run the main thruster, if the lander is falling too fast
    if observation[Obs.VY.value] < -.4:  
        return Act.MAIN.value

    return Act.NO_OP.value 

In [30]:
env = dr.gym_make('LunarLander-v3', 'LL2', render_fps=30)
run_episode(rocket_agent_function, env, verbose = False)
env.close()

dr.show('LL2')

## Evaluating the Agent

Run the agent on 100 problems and report the average reward.

In [31]:
def run_episodes(agent_function, env, n=1000):
    """Run multiple episodes with the given agent and return the rewards for each episode."""
    return [run_episode(agent_function, env, verbose=False, render=False) for _ in range(n)]

Run experiments.

In [32]:
env = gym.make("LunarLander-v3", render_mode=None)

rewards = run_episodes(rocket_agent_function, env)
print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

Average reward: -99.0
Success rate: 5/1000


This is not great performance!


## Implement A Better Reflex-Based Agent

Build a better that uses its right and left thrusters to land the craft (more) safely. Test your agent function using 100 problems.

In [33]:
def improved_agent_function(observation):
    x = observation[Obs.X.value]
    y = observation[Obs.Y.value]
    vx = observation[Obs.VX.value]
    vy = observation[Obs.VY.value]
    angle = observation[Obs.ANGLE.value]
    angular_velocity = observation[Obs.ANGULAR_VELOCITY.value]
    left_leg = observation[Obs.LEFT_LEG_CONTACT.value]
    right_leg = observation[Obs.RIGHT_LEG_CONTACT.value]

    # desired tilt angle: lean towards the pad, counteracting horizontal drift
    target_angle = np.clip(x * 0.5 + vx * 1.0, -0.4, 0.4)
    # desired altitude "cushion": hover a bit higher the further we are from the pad
    target_hover = 0.55 * abs(x)

    angle_error = (target_angle - angle) * 0.5 - angular_velocity * 1.0
    hover_error = (target_hover - y) * 0.5 - vy * 0.5

    if left_leg or right_leg:
        # already touching down: just cushion the descent, don't fight the angle anymore
        angle_error = 0
        hover_error = -vy * 0.5

    if hover_error > abs(angle_error) and hover_error > 0.05:
        return Act.MAIN.value
    elif angle_error < -0.05:
        return Act.RIGHT.value
    elif angle_error > 0.05:
        return Act.LEFT.value
    return Act.NO_OP.value


# quick visual check with the video recorder, same as for the earlier agents
env = dr.gym_make('LunarLander-v3', 'LL3', render_fps=30)
run_episode(improved_agent_function, env, verbose=False)
env.close()
dr.show('LL3')

In [34]:
# Evaluate over 100 episodes, same as we did for the earlier agents
env = gym.make("LunarLander-v3", render_mode=None)

rewards_improved = run_episodes(improved_agent_function, env)

print(f"Improved agent   - Average reward: {np.average(rewards_improved):.2f}, "
      f"Success rate: {np.sum(np.array(rewards_improved) == 100)}/{len(rewards_improved)}")
print(f"Rocket agent     - Average reward: {np.average(rewards):.2f}, "
      f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

Improved agent   - Average reward: 84.80, Success rate: 912/1000
Rocket agent     - Average reward: -99.00, Success rate: 5/1000


## Discussion

Using all three thrusters and combining several observation features into two continuous "error" signals
(instead of a single hard threshold on vertical speed) makes a large difference: the improved agent lands
successfully far more often and with a much higher average reward than the main-thruster-only agent, because
it actively steers the craft horizontally towards the pad and keeps it upright, rather than only slowing its
fall in place.

**Why a purely local, rule-based reflex agent can still struggle in a continuous environment.** Even this
improved version is fundamentally reactive: its action depends only on the *current* observation, with no
memory of past states and no ability to plan ahead or simulate the consequences of an action before taking
it. In a continuous-state, continuous-dynamics environment like this one:

* Small differences in timing (exactly when a thruster fires, for how long, relative to the craft's current
  momentum) compound over many steps, so a rule tuned to work well in one region of the state space (e.g.,
  falling slowly, nearly upright) can behave poorly in another (e.g., falling fast while also tilted and
  drifting sideways), since hand-picked thresholds and gains cannot be optimal everywhere at once.
* The agent cannot look ahead — it reacts to where it currently is, not to where its current trajectory will
  put it a few steps from now, so it may correct a problem (e.g., excessive tilt) only after it has already
  become large, using more fuel and precision than a plan-ahead controller would need.
* There is an inherent coupling between the two controlled quantities (angle and altitude/velocity) that a
  simple reflex rule can only approximate with fixed weights; a small mistake in one dimension (e.g., an
  over-correction on angle) can throw off the other (e.g., cause unwanted horizontal drift), and the reflex
  agent has no way to jointly re-plan both at once.

This is exactly the kind of scenario where reinforcement learning (which we will cover later in the course)
has an advantage: it can learn a much finer-grained, globally-tuned control policy directly from experience,
rather than relying on a person to hand-design and tune a small number of threshold-based rules.